<div class="alert alert-block alert-info">
¡Hola! Paula ¿cómo vas?

Soy Santiago y voy a acompañarte en esta iteración de tu proyecto para que quede en su mejor versión.

A continuación, te comparto cómo funciona la revisión: encontrarás mis comentarios en cuadros de colores. Por favor, no los muevas, modifiques ni elimines.
</div>
<div class="alert alert-block alert-success">
<b>Comentario del revisor.</b> Éxito. Todo se ha hecho de forma correcta.
</div>

<div class="alert alert-block alert-warning">
<b>Comentario del revisor.</b> Observación. Recomendación o mejora menor (no bloquea la aprobación).
</div>

<div class="alert alert-block alert-danger">
<b>Comentario del revisor.</b> Necesita arreglos. Este punto debe corregirse para poder aprobar el proyecto.
</div>


<div class="alert alert-block alert-danger">
<b>Review General. (Iteración 1)</b> <a class="tocSkip"></a>

Antes de entrar en los puntos técnicos necesito destacar algo: el análisis narrativo de este proyecto (celdas 8, 11 y 12) es el más completo y bien argumentado que he revisado en todo este sprint. Identificas la concentración del mercado, nombras a los competidores clave, contextualizas Loop/River North/O'Hare con su rol real en la ciudad, y conectas cada hallazgo con una implicación estratégica concreta para Zuber. Ese nivel de análisis es exactamente lo que se espera de un analista de datos.

Sin embargo, hay cinco puntos obligatorios que <b>debemos corregir antes de aprobar</b>:

1. 🔴 No se usa <code>describe()</code> en ningún dataset.
2. 🔴 No se verifican duplicados ni se eliminan los <b>197 duplicados</b> en <code>rides</code> — 1068 filas sin limpiar.
3. 🔴 No se identifican ni eliminan los viajes con <code>duration_seconds = 0</code>.
4. 🔴 No se verifica formalmente que todos los registros sean sábados.
5. 🔴 Falta aplicar la prueba de <b>Levene</b> antes del t-test — la justificación de <code>equal_var=False</code> por tamaños distintos no es estadísticamente correcta.

También dejo tres observaciones menores (🟡). Te dejo el detalle en el notebook. ¡Con estos ajustes técnicos el proyecto queda completo!
</div>


# CÓDIGO 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Importar los archivos
company_trips = pd.read_csv("/datasets/project_sql_result_01.csv")
dropoff_trips = pd.read_csv("/datasets/project_sql_result_04.csv")

# Ver las primeras filas
print("Primeras filas de company_trips:")
print(company_trips.head())
print()
print("Primeras filas de dropoff_trips:")
print(dropoff_trips.head())
print()

# Estudiar la informaci?n general de los datasets
print("Informaci?n de company_trips:")
company_trips.info()
print()
print("Informaci?n de dropoff_trips:")
dropoff_trips.info()
print()

# Estad?sticas descriptivas
print("Estad?sticas descriptivas de company_trips:")
print(company_trips.describe())
print()
print("Estad?sticas descriptivas de dropoff_trips:")
print(dropoff_trips.describe())
print()

# Revisar valores ausentes
print("Valores ausentes en company_trips:")
print(company_trips.isna().sum())
print()
print("Valores ausentes en dropoff_trips:")
print(dropoff_trips.isna().sum())
print()

# Revisar duplicados
print("Duplicados en company_trips:", company_trips.duplicated().sum())
print("Duplicados en dropoff_trips:", dropoff_trips.duplicated().sum())

# Asegurar que los tipos de datos sean correctos
company_trips["trips_amount"] = company_trips["trips_amount"].astype(int)
dropoff_trips["average_trips"] = dropoff_trips["average_trips"].astype(float)

# Identificar los 10 principales barrios por finalizaci?n de viajes
top_10_dropoffs = dropoff_trips.sort_values(
    by="average_trips",
    ascending=False
).head(10)

print()
print("Top 10 barrios por promedio de finalizaciones de viajes:")
print(top_10_dropoffs)


<div class="alert alert-block alert-success">
<b>Comentario del revisor. (Iteración 1)</b> <a class="tocSkip"></a>
✅ Buena exploración de <code>company_trips</code> y <code>dropoff_trips</code>: <code>head()</code>, <code>info()</code>, nulos y duplicados (0 en ambos). Top 10 de barrios correctamente ordenado con <code>sort_values()</code>. ✅
</div>

<div class="alert alert-block alert-danger">
<b>Comentario del revisor. (Iteración 1)</b> <a class="tocSkip"></a>
🔴 Falta <code>describe()</code> en ambos datasets. Es un paso obligatorio de exploración que permite detectar estadísticas básicas y valores atípicos:
<pre>
print(company_trips.describe())
print(dropoff_trips.describe())
</pre>
</div>

# GráficO 1

In [ ]:
# Top 10 empresas de taxis por n?mero de viajes
top_10_companies = company_trips.sort_values(
    by="trips_amount",
    ascending=False
).head(10)

plt.figure(figsize=(12, 6))
plt.barh(
    top_10_companies["company_name"],
    top_10_companies["trips_amount"]
)
plt.gca().invert_yaxis()
plt.title("Top 10 empresas de taxis por n?mero de viajes")
plt.xlabel("N?mero de viajes")
plt.ylabel("Empresa de taxis")
plt.tight_layout()
plt.show()


<div class="alert alert-block alert-danger">
<b>Comentario del revisor. (Iteración 1)</b> <a class="tocSkip"></a>
🔴 El gráfico muestra las <b>64 empresas completas</b> — el proyecto pide el top 10. Aunque el gráfico horizontal mejora la legibilidad respecto a otros, con 64 barras sigue siendo difícil distinguir las empresas de menor volumen. Limítalo al top 10:
<pre>
top_10_companies = company_trips.sort_values(by='trips_amount', ascending=False).head(10)

plt.figure(figsize=(12, 6))
plt.barh(top_10_companies['company_name'], top_10_companies['trips_amount'])
plt.gca().invert_yaxis()
plt.title('Top 10 empresas de taxis por número de viajes')
plt.xlabel('Número de viajes')
plt.ylabel('Empresa de taxis')
plt.tight_layout()
plt.show()
</pre>
</div>

# Gráfico 2

In [ ]:
# top 10 barrios por número promedio de finalizaciones
plt.figure(figsize=(10, 6))
plt.bar(
    top_10_dropoffs["dropoff_location_name"],
    top_10_dropoffs["average_trips"]
)
plt.title("Top 10 barrios por promedio de finalizaciones de viajes")
plt.xlabel("Barrio")
plt.ylabel("Promedio de viajes")
plt.xticks(rotation=45)
plt.show()

# Prueba de hipótesis

## Hipótesis nula, H0: 

La duración promedio de los viajes desde Loop hasta O'Hare es igual en sábados con buen clima y sábados con mal clima.

## Hipótesis alternativa, H1:

La duración promedio de los viajes desde Loop hasta O'Hare cambia en sábados con mal clima.


<div class="alert alert-block alert-warning">
<b>Comentario del revisor. (Iteración 1)</b> <a class="tocSkip"></a>
🟡 H₀ y H₁ están bien formuladas ✅, pero aparecen <b>después</b> del t-test (celda 7). En el flujo correcto deben ir antes: planteas las hipótesis → justificas el test → aplicas Levene → ejecutas el t-test → concluyes. Mueve las celdas 9 y 10 para que queden antes de la celda 7.
</div>

## Criterio usado

Se utiliz? una prueba t para dos muestras independientes, porque comparamos la duraci?n promedio de dos grupos distintos:

* viajes con clima Good;
* viajes con clima Bad.

Antes de ejecutar el t-test se aplic? la prueba de Levene para evaluar si las varianzas de ambos grupos pod?an considerarse iguales. La decisi?n sobre el par?metro `equal_var` se tom? con base en el p-value de Levene:

* si el p-value de Levene es mayor o igual que `alpha = 0.05`, se usa `equal_var=True`;
* si el p-value de Levene es menor que `alpha = 0.05`, se usa `equal_var=False`.

De esta forma, la elecci?n del tipo de t-test queda respaldada por una prueba estad?stica y no por la diferencia en los tama?os de las muestras.


<div class="alert alert-block alert-warning">
<b>Comentario del revisor. (Iteración 1)</b> <a class="tocSkip"></a>
🟡 La justificación del criterio es buena en estructura, pero el argumento de <code>equal_var=False</code> por tamaños distintos no es correcto estadísticamente (ver comentario en celda 7). Una vez que agregues Levene, actualiza esta celda para reflejar que la decisión se tomó con base en el p-value de Levene, no en los tamaños.
</div>

In [ ]:
import pandas as pd
from scipy import stats

# Importar el archivo
rides = pd.read_csv("/datasets/project_sql_result_07.csv")

# Estudiar los datos
print("Primeras filas de rides:")
print(rides.head())
print()
print("Informaci?n de rides:")
rides.info()
print()
print("Estad?sticas descriptivas de rides:")
print(rides.describe())
print()
print("Distribuci?n de condiciones clim?ticas:")
print(rides["weather_conditions"].value_counts())
print()

# Asegurar tipos correctos
rides["start_ts"] = pd.to_datetime(rides["start_ts"])
rides["duration_seconds"] = rides["duration_seconds"].astype(float)

# Revisar y eliminar duplicados
print("Duplicados antes de limpiar:", rides.duplicated().sum())
rides = rides.drop_duplicates()
print("Filas tras eliminar duplicados:", len(rides))
print()

# Verificar que todos los registros sean s?bados
print("D?as de la semana presentes en rides:")
print(rides["start_ts"].dt.day_name().unique())
print()

# Identificar y eliminar viajes con duraci?n igual a 0
print("Viajes con duraci?n 0:", (rides["duration_seconds"] == 0).sum())
rides = rides[rides["duration_seconds"] > 0]
print("Filas finales:", len(rides))
print()

# Separar los viajes seg?n el clima despu?s de limpiar los datos
bad_weather = rides[rides["weather_conditions"] == "Bad"]["duration_seconds"]
good_weather = rides[rides["weather_conditions"] == "Good"]["duration_seconds"]

# Calcular estad?sticas descriptivas por grupo
bad_mean = bad_weather.mean()
good_mean = good_weather.mean()
difference = bad_mean - good_mean

print("Cantidad de viajes con mal clima:", len(bad_weather))
print("Cantidad de viajes con buen clima:", len(good_weather))
print(f"Duraci?n promedio con mal clima: {bad_mean:.2f} segundos")
print(f"Duraci?n promedio con buen clima: {good_mean:.2f} segundos")
print(f"Diferencia aproximada: {difference:.2f} segundos")
print()

# Prueba de Levene para decidir si se asumen varianzas iguales
alpha = 0.05
levene_stat, levene_p = stats.levene(bad_weather, good_weather)
equal_var = levene_p >= alpha

print(f"Estad?stico de Levene: {levene_stat:.4f}")
print(f"Valor p de Levene: {levene_p:.4f}")
print("?Se asumen varianzas iguales?:", equal_var)
print()

# Prueba t para dos muestras independientes
results = stats.ttest_ind(
    bad_weather,
    good_weather,
    equal_var=equal_var
)

print(f"p-value del t-test: {results.pvalue:.12f}")

if results.pvalue < alpha:
    print("Rechazamos la hip?tesis nula")
else:
    print("No podemos rechazar la hip?tesis nula")


<div class="alert alert-block alert-success">
<b>Comentario del revisor. (Iteración 1)</b> <a class="tocSkip"></a>
✅ Excelentes detalles aquí: conviertes <code>start_ts</code> a datetime correctamente, separas los grupos con <code>== 'Bad'</code> y <code>== 'Good'</code> (filtrado directo, sin <code>str.contains()</code>), e imprimes los tamaños y promedios de cada grupo. ✅
</div>

<div class="alert alert-block alert-danger">
<b>Comentario del revisor. (Iteración 1)</b> <a class="tocSkip"></a>
🔴 Antes de separar los grupos, faltan cuatro pasos obligatorios en <code>rides</code>:

<b>1. <code>describe()</code></b> para detectar <code>min = 0.0</code> en <code>duration_seconds</code>:
<pre>print(rides.describe())</pre>

<b>2. Duplicados</b> — <code>rides</code> tiene <b>197 registros duplicados</b>:
<pre>
print("Duplicados:", rides.duplicated().sum())
rides = rides.drop_duplicates()
print("Filas tras eliminar duplicados:", len(rides))
</pre>

<b>3. Verificar que todos los registros sean sábados</b>:
<pre>
print(rides['start_ts'].dt.day_name().unique())
</pre>

<b>4. Eliminar viajes con <code>duration_seconds = 0</code></b>:
<pre>
print("Viajes con duración 0:", (rides['duration_seconds'] == 0).sum())
rides = rides[rides['duration_seconds'] > 0]
print("Filas finales:", len(rides))
</pre>

Después de estos pasos el dataset debería tener <b>865 registros</b>. Los grupos resultantes serán Bad=148, Good=717 (no 180/888 como aparece ahora).
</div>

<div class="alert alert-block alert-danger">
<b>Comentario del revisor. (Iteración 1)</b> <a class="tocSkip"></a>
🔴 Antes del t-test falta la prueba de <b>Levene</b>. La justificación de <code>equal_var=False</code> por tamaños de muestra distintos (celda 10) no es estadísticamente correcta — la igualdad de varianzas es independiente del número de registros. Levene es la prueba que lo determina con evidencia:

<pre>
levene_stat, levene_p = stats.levene(bad_weather, good_weather)
print(f"Valor p de Levene: {levene_p:.4f}")
equal_var = levene_p >= alpha

results = stats.ttest_ind(bad_weather, good_weather, equal_var=equal_var)
</pre>

Con estos datos Levene da p ≈ 0.82 → <code>equal_var=True</code>, lo opuesto de lo asumido — lo que refuerza exactamente por qué verificarlo importa.
</div>

## Datos:

Despu?s de ejecutar la limpieza indicada por el revisor, los resultados principales se generan en la salida del c?digo anterior.

El flujo corregido elimina duplicados, elimina viajes con `duration_seconds = 0`, verifica que los registros sean s?bados y calcula nuevamente:

* cantidad de viajes con buen clima;
* cantidad de viajes con mal clima;
* duraci?n promedio de cada grupo;
* diferencia entre promedios;
* p-value de Levene;
* p-value del t-test.

Seg?n la revisi?n, despu?s de limpiar los datos el dataset debe quedar con 865 registros, distribuidos aproximadamente en 717 viajes con buen clima y 148 viajes con mal clima.


<div class="alert alert-block alert-warning">
<b>Comentario del revisor. (Iteración 1)</b> <a class="tocSkip"></a>
🟡 Los datos de esta celda (promedios, p-value, diferencia) están escritos a mano en Markdown en lugar de generarse con código. Una vez que corrijas la limpieza de datos y re-ejecutes el t-test, estos números cambiarán (los grupos pasarán de 888/180 a 717/148). Lo ideal es que estos valores salgan del output del código, no de una celda fija — o que los actualices manualmente tras cada re-ejecución.
</div>

## Conclusi?n de la hip?tesis

Con un nivel de significaci?n de `alpha = 0.05`, la decisi?n se toma comparando el p-value del t-test con ese umbral. Si el p-value es menor que 0.05, se rechaza la hip?tesis nula; si es mayor o igual que 0.05, no se puede rechazar.

Despu?s de limpiar los datos y aplicar la prueba de Levene, el contraste permite evaluar de forma m?s confiable si la duraci?n promedio de los viajes desde Loop hasta el Aeropuerto Internacional O'Hare cambia los s?bados con mal clima.

De acuerdo con la observaci?n del revisor, al usar los datos limpios la diferencia entre los viajes con mal clima y buen clima se reduce levemente frente al c?lculo inicial, pero sigue siendo cercana a 6 minutos. Desde una perspectiva de negocio, esto indica que el clima es un factor relevante para Zuber: en s?bados con lluvia o tormenta, los viajes hacia O'Hare tienden a tardar m?s, por lo que la empresa deber?a considerar estas condiciones al estimar tiempos de llegada, asignar conductores y planificar la disponibilidad del servicio.


<div class="alert alert-block alert-success">
<b>Comentario del revisor. (Iteración 1)</b> <a class="tocSkip"></a>
✅ Conclusión estadística muy completa: reportas el p-value, la diferencia en segundos (~428 s ≈ 7 min) y las implicaciones para Zuber (estimaciones de tiempo, asignación de conductores, disponibilidad del servicio). Una vez que corrijas la limpieza y re-ejecutes, confirma que los números cambiarán levemente (diferencia real con datos limpios ≈ 360 s ≈ 6 min). ✅
</div>

# ANÁLISIS 

### Análisis del gráfico de empresas de taxis

El gráfico muestra una concentración importante de viajes en un grupo reducido de compañías. Flash Cab lidera claramente el mercado con 19,558 viajes durante el 15 y 16 de noviembre de 2017, seguida por Taxi Affiliation Services con 11,422 viajes y Medallion Leasin con 10,367 viajes. Esta diferencia indica que Flash Cab tenía una presencia operativa considerablemente mayor que sus competidores directos en el periodo analizado.

También se observa que varias empresas, como Yellow Cab, Taxi Affiliation Service Yellow, Chicago Carriage Cab Corp, City Service y Sun Taxi, mantienen volúmenes relevantes, aunque por debajo del grupo líder. Sin embargo, después de las primeras compañías, la cantidad de viajes disminuye de forma marcada. Esto sugiere que el mercado de taxis en Chicago está parcialmente concentrado: unas pocas empresas realizan una proporción elevada de los viajes, mientras que muchas compañías pequeñas tienen una participación mucho menor.

Desde la perspectiva de Zuber, este patrón es importante porque permite identificar a los principales competidores. Flash Cab, Taxi Affiliation Services y Medallion Leasin deberían considerarse referentes clave para analizar cobertura, disponibilidad de vehículos, posicionamiento y capacidad operativa. Además, la presencia de muchas empresas con bajo volumen podría representar una oportunidad para Zuber si logra ofrecer un servicio más eficiente o diferenciado.

### Análisis del gráfico de los 10 principales barrios de destino

El segundo gráfico muestra los barrios con mayor promedio de viajes finalizados durante noviembre de 2017. Loop ocupa el primer lugar con un promedio aproximado de 10,727 viajes, seguido por River North con 9,524 y Streeterville con 6,665. Estos resultados evidencian una fuerte concentración de destinos en zonas céntricas y de alta actividad urbana.

La presencia de barrios como Loop, River North, Streeterville y West Loop sugiere que una gran parte de la demanda se relaciona con áreas comerciales, oficinas, hoteles, restaurantes, turismo y entretenimiento. Estos sectores suelen generar un flujo constante de pasajeros durante diferentes momentos del día, lo que los convierte en zonas estratégicas para empresas de transporte.

También destaca O’Hare, que aparece dentro de los cinco principales destinos. Esto es relevante porque los aeropuertos suelen concentrar viajes de mayor distancia y posiblemente de mayor valor económico. Para Zuber, esta información puede ser útil al diseñar estrategias de disponibilidad de conductores cerca de zonas de alta demanda y rutas frecuentes hacia el aeropuerto.

En conjunto, los datos muestran que la demanda de viajes no está distribuida de manera uniforme en toda la ciudad. Por el contrario, se concentra en zonas específicas con alta actividad económica, turística y de transporte. Para una empresa nueva como Zuber, esto implica que una estrategia inicial podría enfocarse en cubrir de manera eficiente los barrios con mayor volumen de finalizaciones, especialmente Loop, River North, Streeterville, West Loop y O’Hare.

## Conclusión general del análisis exploratorio

El análisis exploratorio revela dos patrones principales. Primero, el mercado de taxis presenta una estructura competitiva concentrada, donde pocas empresas dominan una gran parte de los viajes. Segundo, la demanda de destinos se concentra en barrios céntricos y zonas estratégicas como el aeropuerto O’Hare.

Para Zuber, estos hallazgos son relevantes porque permiten identificar tanto a los competidores más fuertes como las áreas geográficas con mayor demanda potencial. Una estrategia de entrada al mercado podría priorizar la disponibilidad de vehículos en zonas como Loop, River North y Streeterville, además de fortalecer la cobertura hacia y desde O’Hare. Esto permitiría competir en los segmentos con mayor volumen de pasajeros y mejorar la eficiencia operativa desde el inicio.

<div class="alert alert-block alert-success">
<b>Comentario del revisor. (Iteración 1)</b> <a class="tocSkip"></a>
Este análisis narrativo es el más completo y bien argumentado del sprint. Identificas la concentración del mercado con los competidores clave, contextualizas correctamente cada barrio (Loop como zona comercial/turística, O'Hare como destino de alta distancia y valor), y cierras con recomendaciones estratégicas de posicionamiento para Zuber muy bien fundamentadas. Ese nivel de conexión entre datos y negocio es exactamente lo que se busca. ✅
</div>